[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S08/S08_05_busqueda_hiperparametros.ipynb)

# S08 · 05 — Buscar buenos hiperparámetros: GridSearchCV y RandomizedSearchCV

**Tiempo estimado:** 60 min · **Sprint 8** · M3 (UD 3.2 y 3.6–3.8)

**Contexto TurisData.** Hemos visto que cada modelo tiene «botones» (`max_depth`, `learning_rate`, `C`…).
Girarlos a mano es lento y sesgado. Los **buscadores** los prueban de forma sistemática con
validación cruzada. Pero ojo: **buscar mucho también puede engañarnos**, y el test debe seguir siendo un
examen que se hace **una sola vez**.

**Al terminar sabrás:**
1. Diferenciar **parámetros** (los aprende el modelo) de **hiperparámetros** (los fijamos nosotros).
2. Usar `GridSearchCV` (rejilla completa) y `RandomizedSearchCV` (muestreo aleatorio) y leer `cv_results_`.
3. Elegir entre configuraciones casi empatadas (regla de «1 error estándar»).
4. Evaluar el modelo final en test **una sola vez** y entender por qué la nota de la búsqueda es optimista.

In [ ]:
# Preparación (ejecuta esta celda siempre la primera)
import os, urllib.request

BASE_URL = "https://raw.githubusercontent.com/cCoolBoox/IFCD107-alumnado/main/datos/"

def dato(nombre):
    """Devuelve la ruta local de un archivo de datos del curso (lo descarga si no está)."""
    for carpeta in ("datos", "../datos", "../../datos"):
        ruta = os.path.join(carpeta, nombre)
        if os.path.exists(ruta):
            return ruta
    os.makedirs("datos", exist_ok=True)
    ruta = os.path.join("datos", nombre)
    urllib.request.urlretrieve(BASE_URL + nombre, ruta)
    return ruta


In [ ]:
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import randint, loguniform, uniform
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score
import lightgbm as lgb

warnings.filterwarnings("ignore")
SEMILLA = 42
sns.set_theme(style="whitegrid")

df = pd.read_csv(dato("reservas_turisdata.csv"), parse_dates=["fecha_reserva", "fecha_llegada"])
df["temporada_alta"] = df["fecha_llegada"].dt.month.isin([1, 2, 3, 7, 8, 12]).astype(int)
columnas_num = ["antelacion_dias", "noches", "adultos", "ninos", "precio_noche",
                "peticiones_especiales", "cliente_repetidor", "cancelaciones_previas", "temporada_alta"]
columnas_cat = ["tipo_habitacion", "regimen", "canal", "pais_origen", "tarifa", "deposito"]
X, y = df[columnas_num + columnas_cat], df["cancelada"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEMILLA, stratify=y)
cv = StratifiedKFold(3, shuffle=True, random_state=SEMILLA)

prep = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), columnas_num),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                      ("oh", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), columnas_cat),
])

## 1. Parámetros frente a hiperparámetros

| | Parámetros | Hiperparámetros |
|---|---|---|
| Quién los fija | El **modelo**, al entrenar | **Nosotros**, antes de entrenar |
| Ejemplos | Coeficientes de la logística, umbrales de un árbol | `max_depth`, `learning_rate`, `C`, `k`, `alpha` |

Un buscador prueba combinaciones de hiperparámetros, evalúa cada una con **validación cruzada** (sobre train)
y se queda con la mejor (`refit=True` reentrena con ella todo el train).

- **`GridSearchCV`**: prueba **todas** las combinaciones de una rejilla. Exhaustivo, pero se dispara:
  3 valores × 3 valores × 3 valores × 5 folds = 135 entrenamientos.
- **`RandomizedSearchCV`**: prueba `n_iter` combinaciones **al azar** de distribuciones. Explora más
  valores distintos de los hiperparámetros importantes con el mismo presupuesto.

In [ ]:
# Por qué el aleatorio explora mejor: 9 pruebas en rejilla vs 9 al azar (dos hiperparámetros)
rng = np.random.default_rng(SEMILLA)
gx, gy = np.meshgrid([0.2, 0.5, 0.8], [0.2, 0.5, 0.8])
fig, ejes = plt.subplots(1, 2, figsize=(9, 3.6), sharey=True)
ejes[0].scatter(gx.ravel(), gy.ravel(), color="#2a6f97")
ejes[0].set(title="Rejilla: 9 pruebas, solo 3 valores de cada uno", xlabel="Hiperparámetro importante", ylabel="Hiperparámetro poco importante")
ejes[1].scatter(rng.random(9), rng.random(9), color="#e63946")
ejes[1].set(title="Aleatorio: 9 pruebas, 9 valores de cada uno", xlabel="Hiperparámetro importante")
plt.tight_layout()
plt.show()

## 2. `GridSearchCV` con un bosque aleatorio

**Ejercicio 1.** Define un `Pipeline` `pipe_rf` con `prep` y `RandomForestClassifier(random_state=SEMILLA, n_jobs=2)`
(paso llamado `"rf"`) y una rejilla:
- `n_estimators`: `[100, 200]`,
- `max_depth`: `[4, 8]`,
- `min_samples_leaf`: `[10, 40]`.

Crea `busqueda_grid = GridSearchCV(pipe_rf, rejilla, cv=cv, scoring="roc_auc")`, ajústala con **train** y guarda
en `n_combinaciones` cuántas combinaciones hay (con `len(busqueda_grid.cv_results_["params"])`) y en `mejor_auc_grid`
su `best_score_`.

In [ ]:
# TODO: los nombres de la rejilla llevan el prefijo del paso: "rf__max_depth", etc.
...
assert n_combinaciones == 8, "2 × 2 × 2 = 8 combinaciones"
assert 0.65 < mejor_auc_grid < 0.85, "El AUC de validación debería estar entre 0,65 y 0,85"
print(f"{n_combinaciones} combinaciones × 3 folds = {n_combinaciones * 3} entrenamientos en {segundos_grid:.0f} s")
print("Mejores parámetros:", busqueda_grid.best_params_, f"· AUC CV = {mejor_auc_grid:.3f}")

## 3. `RandomizedSearchCV` con LightGBM

En lugar de listas, damos **distribuciones** (`scipy.stats`): `randint(a, b)` para enteros y
`loguniform(a, b)` para valores que varían en órdenes de magnitud (como el `learning_rate`).

**Ejercicio 2.** Crea `pipe_lgb` (`prep` + `lgb.LGBMClassifier(n_estimators=200, random_state=SEMILLA, n_jobs=2, verbose=-1)`,
paso `"lgb"`) y una búsqueda `busqueda_rand = RandomizedSearchCV(pipe_lgb, distribuciones, n_iter=25, cv=cv,
scoring="roc_auc", random_state=SEMILLA)` con estas `distribuciones`:
- `lgb__learning_rate`: `loguniform(0.01, 0.3)`,
- `lgb__num_leaves`: `randint(3, 40)`,
- `lgb__min_child_samples`: `randint(10, 200)`,
- `lgb__subsample`: `uniform(0.6, 0.4)` (equivale a valores entre 0,6 y 1,0; recuerda añadir `subsample_freq=1` al modelo).

Ajústala con train y guarda `mejor_auc_rand` (`best_score_`).

In [ ]:
# TODO: RandomizedSearchCV(pipe_lgb, distribuciones, n_iter=25, ...).fit(X_train, y_train)
...
assert len(busqueda_rand.cv_results_["params"]) == 25, "Deben probarse 25 combinaciones"
assert 0.65 < mejor_auc_rand < 0.85
print(f"25 combinaciones aleatorias × 3 folds en {segundos_rand:.0f} s · AUC CV = {mejor_auc_rand:.3f}")
print("Mejores parámetros:", {k: round(v, 3) if isinstance(v, float) else v for k, v in busqueda_rand.best_params_.items()})

## 4. Leer `cv_results_`: ¿hay un ganador claro?

Toda la búsqueda queda en `cv_results_`: para cada combinación, la media y la desviación de la
puntuación en validación cruzada. Es habitual encontrar **muchas configuraciones casi empatadas**.
Una regla práctica es la de **«1 error estándar»**: entre las configuraciones cuya media queda dentro de
**una desviación** de la mejor, elegir la **más sencilla** (menos hojas, menos árboles…): rendimiento
equivalente, menos riesgo y menos coste.

**Ejercicio 3.** Con `resultados = pd.DataFrame(busqueda_rand.cv_results_)`, guarda en `top5` las 5 mejores
filas (columnas `param_lgb__num_leaves`, `param_lgb__learning_rate`, `mean_test_score`, `std_test_score`,
ordenadas por `mean_test_score` descendente), en `n_empatadas` cuántas combinaciones tienen una media dentro
de **una desviación** (`std_test_score` de la mejor) de la mejor media, y en `mas_simple` el **menor**
`num_leaves` entre esas empatadas.

In [ ]:
resultados = pd.DataFrame(busqueda_rand.cv_results_)
# TODO: sort_values("mean_test_score", ascending=False).head(5); filtra por mean >= mejor_media - std_del_mejor
...
assert len(top5) == 5 and top5["mean_test_score"].is_monotonic_decreasing, "top5: cinco filas ordenadas de mejor a peor"
assert n_empatadas >= 2, "Suele haber varias configuraciones casi empatadas"
assert 3 <= mas_simple <= 39
print(f"{n_empatadas} de 25 configuraciones están a menos de 1 error estándar de la mejor; la más sencilla usa {mas_simple} hojas.")
top5.round(3)

In [ ]:
plt.figure(figsize=(7, 4))
plt.scatter(resultados["param_lgb__learning_rate"].astype(float), resultados["mean_test_score"], c=resultados["param_lgb__num_leaves"].astype(int),
            cmap="viridis", s=50)
plt.colorbar(label="num_leaves")
plt.xscale("log")
plt.title("LightGBM: AUC (validación cruzada) según learning_rate")
plt.xlabel("learning_rate (escala logarítmica)")
plt.ylabel("AUC medio en validación cruzada")
plt.show()

## 5. La nota de la búsqueda es optimista

Cuando se prueban muchas configuraciones, la **mejor puntuación de validación** es en parte **suerte**
(la «maldición del ganador»): entre 25 intentos, alguno sale bien por azar. Por eso el resultado honesto
es el del **test**, que no se ha usado para elegir nada y **se evalúa una sola vez** con el modelo final.

**Ejercicio 4.** Con el mejor modelo de cada búsqueda (`busqueda_grid.best_estimator_` y
`busqueda_rand.best_estimator_`, ya reentrenados con todo train), guarda `auc_test_grid` y `auc_test_rand`
(AUC en test con `predict_proba`) y en `optimismo_rand` la diferencia `mejor_auc_rand - auc_test_rand`.

In [ ]:
# TODO: best_estimator_.predict_proba(X_test)[:, 1]
...
assert 0.65 < auc_test_grid < 0.85 and 0.65 < auc_test_rand < 0.85
print(f"Rejilla + RF: AUC CV {mejor_auc_grid:.3f} → test {auc_test_grid:.3f}")
print(f"Aleatorio + LightGBM: AUC CV {mejor_auc_rand:.3f} → test {auc_test_rand:.3f} (diferencia {optimismo_rand:+.3f})")

## 6. Mini-reto: informe de la búsqueda

Completa el diccionario `informe` con estas claves (para el **mejor de los dos buscadores según su AUC en
validación**):
- `"modelo"`: `"random_forest"` o `"lightgbm"`,
- `"parametros"`: los `best_params_` correspondientes,
- `"auc_validacion"` y `"auc_test"`,
- `"n_entrenamientos"`: total de ajustes hechos en su búsqueda (combinaciones × folds),
- `"recomendacion"`: una frase (texto) que diga si el resultado te parece fiable y por qué (menciona la
  diferencia validación–test o los empates).

In [ ]:
# TODO: compara mejor_auc_grid con mejor_auc_rand y rellena el diccionario con los datos del ganador
...
assert set(informe) == {"modelo", "parametros", "auc_validacion", "auc_test", "n_entrenamientos", "recomendacion"}, "Faltan o sobran claves"
assert informe["modelo"] in {"random_forest", "lightgbm"}
assert informe["n_entrenamientos"] in {24, 75}, "8 × 3 = 24 (rejilla) o 25 × 3 = 75 (aleatorio)"
assert isinstance(informe["recomendacion"], str) and len(informe["recomendacion"]) > 30
informe

## 7. Para reflexionar

1. La rejilla probó 8 combinaciones y el aleatorio 25. ¿Cuál «costó» más entrenamientos? ¿Cuál exploró más valores distintos de `learning_rate`?
2. Varias configuraciones quedan a menos de un error estándar de la mejor. ¿Por qué elegir la más sencilla?
3. Si tras buscar 500 configuraciones tu AUC de validación es 0,80 y en test 0,74, ¿qué ha pasado?
4. ¿Por qué el test no se usa nunca para elegir hiperparámetros? ¿Qué harías si necesitaras una estimación aún más honesta (pista: validación cruzada anidada)?

## 8. Resumen: qué has aprendido
- Los hiperparámetros se fijan antes de entrenar; los buscadores los prueban con validación cruzada **sobre train**.
- `GridSearchCV` es exhaustivo (caro); `RandomizedSearchCV` explora más con el mismo presupuesto.
- `cv_results_` muestra que suele haber empates: elige lo **simple** dentro de un error estándar.
- La nota de la búsqueda es optimista: el test se evalúa **una sola vez** al final.